# VisionGuard V3 — Adversarial Computer Vision

Ataques e defesas adversariais sobre a etapa de classificação **ResNet** (V2). A detecção YOLO (V1) e a classificação ResNet (V2) permanecem intactas — a V3 é uma camada de avaliação/hardening por cima.

`nada é simulado`: as acurácias limpa/adversarial vêm da avaliação real do modelo sobre as imagens perturbadas.

## Imports

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import torch
from torchvision import models
from src.adversarial import fgsm, pgd, make_patch, evaluate_robustness, adversarial_gap, adversarial_finetune


## Modelo-alvo: ResNet-18 (mesmo backbone da V2)

`ResNetClassifier` da V2 usa `resnet18` pré-treinado (ImageNet). Aqui carregamos o mesmo backbone e atacamos suas logits diretamente.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights).eval()

# lote de exemplo (substitua por crops reais das detecções do YOLO)
x = torch.rand(8, 3, 224, 224)
with torch.no_grad():
    y = model(x).argmax(1)  # 'rótulo verdadeiro' = predição limpa


## FGSM e PGD

In [ ]:
x_fgsm = fgsm(model, x, y, epsilon=0.03)
x_pgd  = pgd(model, x, y, epsilon=0.03, alpha=0.01, steps=10)

acc = lambda xx: (model(xx).argmax(1) == y).float().mean().item()
print(f'clean : {acc(x):.3f}')
print(f'FGSM  : {acc(x_fgsm):.3f}')
print(f'PGD   : {acc(x_pgd):.3f}')


## Adversarial patch (perturbação localizada e visível)

Modela um adesivo físico: um recorte 24×24 no canto superior esquerdo, otimizado para empurrar a predição para a classe `target`.

In [ ]:
x_patched, patch = make_patch(model, x[:4], target=int(y[0].item()) ^ 1 if False else 5,
                              patch_size=24, top=8, left=8, steps=40)
print('predição no patch:', model(x_patched).argmax(1).tolist())


## Robustez: adversarial vs. corrupção não-adversarial

Se a acurácia cai igual sob ruído gaussiano e sob PGD do mesmo tamanho, o modelo é frágil em geral. Se cai muito mais sob PGD, a fragilidade é **adversarial específica** (`adversarial_gap`).

In [ ]:
res = evaluate_robustness(model, x, y, epsilon=0.03)
for k, v in res.items():
    print(f'{k:16s} {v:.3f}')
print(f'
adversarial gap (ruído - PGD): {adversarial_gap(res):.3f}')


## Adversarial training (fine-tune curto)

`adversarial_finetune` treina em (limpo + PGD) por N épocas. Aqui só um esboço — no uso real, treine sobre um dataset de crops rotulados do domínio.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import copy
m2 = copy.deepcopy(model)
loader = DataLoader(TensorDataset(x, y), batch_size=4, shuffle=True)
adversarial_finetune(m2, loader, epochs=1, epsilon=0.03, alpha=0.01, pgd_steps=3, lr=1e-4)
print('PGD acc antes :', evaluate_robustness(model, x, y, epsilon=0.03)['pgd'])
print('PGD acc depois:', evaluate_robustness(m2, x, y, epsilon=0.03)['pgd'])


## Nota

Para YOLO (detecção), a extensão natural é medir mAP/precision/recall antes/depois de FGSM/PGD/patch — deixada como próximo passo (exige um dataset de detecção rotulado). Este módulo cobre a parte de classificação, que é onde a ResNet da V2 atua.